# 05 — Analyse Prédictive & Détection d'Anomalies
## Helpdesk IA — GEISER | PFE 2026

### Objectif
Implémenter des analyses avancées sur les tickets :
1. **Analyse temporelle** — tendances par jour/heure/mois
2. **Prévision volume** — Prophet (séries temporelles)
3. **Clustering tickets répétitifs** — KMeans + PCA
4. **Détection anomalies** — Isolation Forest
5. **Analyse charge agents** — heatmap par semaine
6. **Prédiction risque SLA** — Gradient Boosting

### Dataset
- **Source** : `dataset_labeled.csv` (produit par notebook 01)
- **Dates** : `data/Ticket_export.csv` (colonnes DATE DE CRÉATION)

### Stack technique
- **Prophet** — prévision séries temporelles (Facebook/Meta)
- **KMeans + PCA** — clustering et visualisation 2D
- **Isolation Forest** — détection d'anomalies
- **Gradient Boosting** — prédiction risque SLA

---
## 1. Installation

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================
import subprocess, sys

packages = ['prophet', 'scikit-learn', 'pandas',
            'matplotlib', 'seaborn', 'plotly']

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip',
                   'install', pkg, '-q'])

print("✅ Dépendances installées !")

---
## 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

print("✅ Imports OK")

---
## 3. Chargement et Préparation des Données Temporelles

In [ ]:
# ============================================================
# CHARGEMENT DONNÉES
# ============================================================

# Dataset brut — pour les dates (DATE DE CRÉATION)
df_raw = pd.read_csv('../../data/Ticket_export.csv',
                     encoding='utf-8', sep=None, engine='python')

# Dataset labeled — pour les catégories, priorités, sous-catégories
df_clean = pd.read_csv('../../data/dataset_labeled.csv',
                       encoding='utf-8')

# Parser les dates
df_raw['date_creation'] = pd.to_datetime(
    df_raw['DATE DE CRÉATION'], format='mixed', dayfirst=False
)

print(f"✅ Dataset brut   : {len(df_raw)} tickets")
print(f"✅ Dataset labeled: {len(df_clean)} tickets")
print(f"\nPériode :")
print(f"  Début  : {df_raw['date_creation'].min()}")
print(f"  Fin    : {df_raw['date_creation'].max()}")
print(f"  Durée  : {(df_raw['date_creation'].max() - df_raw['date_creation'].min()).days} jours")
print(f"\nCatégories disponibles : {df_clean['categorie'].unique().tolist()}")

---
## 4. Analyse Temporelle Exploratoire
### Distribution par jour, semaine, mois

In [ ]:
# ============================================================
# ANALYSE TEMPORELLE
# Visualisation de la distribution des tickets par :
#   - Jour de la semaine
#   - Heure de la journée
#   - Mois
#   - Semaine
# ============================================================
df_raw['jour_semaine'] = df_raw['date_creation'].dt.day_name()
df_raw['heure']        = df_raw['date_creation'].dt.hour
df_raw['mois']         = df_raw['date_creation'].dt.to_period('M')
df_raw['semaine']      = df_raw['date_creation'].dt.to_period('W')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Analyse Temporelle des Tickets — GEISER', fontsize=14, fontweight='bold')

jours_ordre = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
jours_fr    = ['Lundi','Mardi','Mercredi','Jeudi','Vendredi','Samedi','Dimanche']

jour_counts = df_raw['jour_semaine'].value_counts().reindex(jours_ordre, fill_value=0)
axes[0,0].bar(jours_fr, jour_counts.values, color='#534AB7')
axes[0,0].set_title('Tickets par jour de la semaine')
axes[0,0].set_xlabel('Jour')
axes[0,0].set_ylabel('Nb tickets')
axes[0,0].tick_params(axis='x', rotation=45)

heure_counts = df_raw['heure'].value_counts().sort_index()
axes[0,1].plot(heure_counts.index, heure_counts.values, color='#1D9E75', marker='o', linewidth=2)
axes[0,1].fill_between(heure_counts.index, heure_counts.values, alpha=0.3, color='#1D9E75')
axes[0,1].set_title('Tickets par heure de la journée')
axes[0,1].set_xlabel('Heure')
axes[0,1].set_ylabel('Nb tickets')
axes[0,1].set_xticks(range(0, 24, 2))

mois_counts = df_raw.groupby('mois').size()
axes[1,0].bar(range(len(mois_counts)), mois_counts.values, color='#185FA5')
axes[1,0].set_title('Évolution mensuelle des tickets')
axes[1,0].set_xlabel('Mois')
axes[1,0].set_ylabel('Nb tickets')
axes[1,0].set_xticks(range(len(mois_counts)))
axes[1,0].set_xticklabels([str(m) for m in mois_counts.index], rotation=45)

semaine_counts = df_raw.groupby('semaine').size()
axes[1,1].plot(range(len(semaine_counts)), semaine_counts.values, color='#D85A30', linewidth=2)
axes[1,1].fill_between(range(len(semaine_counts)), semaine_counts.values, alpha=0.3, color='#D85A30')
axes[1,1].set_title('Évolution hebdomadaire')
axes[1,1].set_xlabel('Semaine')
axes[1,1].set_ylabel('Nb tickets')

plt.tight_layout()
plt.savefig('../../diagrams/analyse_temporelle.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Pic d'activité :")
print(f"   Jour le plus chargé : {jours_fr[list(jours_ordre).index(jour_counts.idxmax())]}")
print(f"   Heure de pointe     : {heure_counts.idxmax()}h")

print(f"\n📅 Volume mensuel :")
print(mois_counts)
print(f"\n📆 Période totale : {(df_raw['date_creation'].max() - df_raw['date_creation'].min()).days} jours")


---
## 5. Prévision du Volume de Tickets (Prophet)

**Prophet** est un outil de Facebook/Meta pour la
prévision de séries temporelles.

### Pourquoi Prophet ?
- Gère les tendances et saisonnalités automatiquement
- Robuste aux valeurs manquantes
- Interprétable et visualisable facilement

In [ ]:
# ============================================================
# PRÉVISION VOLUME — PROPHET
# Prophet prédit le nombre de tickets par jour pour les
# 30 prochains jours en tenant compte des saisonnalités
# hebdomadaire et annuelle.
# ============================================================
from prophet import Prophet

# Préparer données pour Prophet (format obligatoire : ds, y)
df_prophet = df_raw.groupby(df_raw['date_creation'].dt.date).size().reset_index()
df_prophet.columns = ['ds', 'y']
df_prophet['ds']   = pd.to_datetime(df_prophet['ds'])

print(f"✅ Données Prophet : {len(df_prophet)} jours")
print(df_prophet.tail(5))

# Entraîner Prophet
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
model_prophet.fit(df_prophet)

# Prévision 30 jours futurs
future   = model_prophet.make_future_dataframe(periods=30)
forecast = model_prophet.predict(future)

# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(df_prophet['ds'], df_prophet['y'],
             'o', color='#534AB7', alpha=0.5, markersize=3, label='Réel')
axes[0].plot(forecast['ds'], forecast['yhat'],
             '-', color='#1D9E75', linewidth=2, label='Prévision')
axes[0].fill_between(forecast['ds'],
                     forecast['yhat_lower'], forecast['yhat_upper'],
                     alpha=0.2, color='#1D9E75', label='Intervalle confiance')
axes[0].axvline(x=df_prophet['ds'].max(),
                color='red', linestyle='--', alpha=0.7, label="Aujourd'hui")
axes[0].set_title('Prévision Volume Tickets (Prophet) — 30 jours', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Nb tickets / jour')
axes[0].legend()

axes[1].plot(forecast['ds'][:len(df_prophet)],
             forecast['weekly'][:len(df_prophet)], color='#D85A30', linewidth=1.5)
axes[1].set_title('Saisonnalité hebdomadaire')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Effet saisonnier')

plt.tight_layout()
plt.savefig('../../diagrams/prevision_prophet.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📈 PRÉVISIONS 10 PROCHAINS JOURS :")
future_only = forecast[forecast['ds'] > df_prophet['ds'].max()][
    ['ds', 'yhat', 'yhat_lower', 'yhat_upper']
].head(10)
print(future_only.to_string(index=False))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Backtest : on retire les 30 derniers jours pour servir de test
df_train_bt = df_prophet.iloc[:-30]
df_test_bt  = df_prophet.iloc[-30:]

model_bt = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
model_bt.fit(df_train_bt)

future_bt   = model_bt.make_future_dataframe(periods=30)
forecast_bt = model_bt.predict(future_bt)

y_true = df_test_bt['y'].values
y_pred = forecast_bt['yhat'].tail(30).values

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Backtest Prophet (30 derniers jours retenus comme test) :")
print(f"MAE  : {mae:.2f} tickets/jour")
print(f"RMSE : {rmse:.2f} tickets/jour")


---
## 6. Clustering des Tickets Répétitifs (KMeans)

Identifier les groupes de tickets similaires
pour détecter les problèmes récurrents.

### Méthode
1. Vectorisation TF-IDF des textes
2. Réduction dimensionnelle PCA
3. Clustering KMeans
4. Analyse des clusters

In [ ]:
# ============================================================
# MÉTHODE ELBOW — Trouver le nombre optimal de clusters K
# On entraîne KMeans avec différentes valeurs de K et on
# choisit le "coude" de la courbe d'inertie.
# ============================================================
texts = df_clean['text'].fillna('').tolist()

# Vectorisation TF-IDF
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(texts)

# Calcul inertie pour K = 2 à 14
inertias = []
K_range  = range(2, 15)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_tfidf)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8, color='#534AB7')
plt.xlabel('Nombre de clusters (K)')
plt.ylabel('Inertie')
plt.title('Méthode Elbow — Nombre optimal de clusters KMeans')
plt.grid(True, alpha=0.3)

diff     = np.diff(inertias)
diff2    = np.diff(diff)
optimal_k = K_range[np.argmax(diff2) + 1]
plt.axvline(x=optimal_k, color='red', linestyle='--', label=f'K optimal = {optimal_k}')
plt.legend()
plt.tight_layout()
plt.savefig('../../diagrams/elbow_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ K optimal suggéré : {optimal_k} clusters")

In [ ]:
# ============================================================
# CLUSTERING KMEANS
# Chaque ticket est assigné à un cluster.
# On analyse les mots-clés et catégories dominantes
# de chaque cluster pour comprendre les thèmes récurrents.
# ============================================================
N_CLUSTERS = optimal_k

kmeans           = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_clean         = df_clean.copy()
df_clean['cluster'] = kmeans.fit_predict(X_tfidf)

print(f"=== ANALYSE DES {N_CLUSTERS} CLUSTERS ===\n")

for c in range(N_CLUSTERS):
    df_c        = df_clean[df_clean['cluster'] == c]
    size        = len(df_c)
    center      = kmeans.cluster_centers_[c]
    top_indices = center.argsort()[-8:][::-1]
    keywords    = [tfidf.get_feature_names_out()[i] for i in top_indices]
    top_cat     = df_c['categorie'].value_counts().index[0]
    top_cat_pct = df_c['categorie'].value_counts().iloc[0] / size * 100

    print(f"Cluster {c:2d} ({size:4d} tickets) | {top_cat:<22} ({top_cat_pct:.0f}%)")
    print(f"   Mots-clés : {', '.join(keywords[:5])}")
    print()

# Visualisation PCA 2D
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tfidf.toarray())

plt.figure(figsize=(14, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df_clean['cluster'], cmap='tab10', alpha=0.6, s=20)
plt.colorbar(scatter, label='Cluster')
plt.title(f'Clusters KMeans — Visualisation PCA 2D\n{N_CLUSTERS} clusters, {len(texts)} tickets',
          fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.tight_layout()
plt.savefig('../../diagrams/clusters_pca.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score
sil_score = silhouette_score(X_tfidf, df_clean['cluster'])
print(f"Silhouette Score (K={N_CLUSTERS}) : {sil_score:.4f}")

In [ ]:
# ============================================================
# SAUVEGARDE MODÈLE CLUSTERING — pour utilisation dans la plateforme
# ============================================================
import pickle
import os

save_dir = '../app/ai_service/models'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/model_cluster.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
with open(f'{save_dir}/tfidf_cluster.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("✅ Modèle clustering sauvegardé : model_cluster.pkl, tfidf_cluster.pkl")

---
## 7. Détection d'Anomalies (Isolation Forest)

**Isolation Forest** détecte les tickets anormaux :
- Tickets avec description inhabituelle
- Tickets qui ne ressemblent à aucun autre
- Potentiels tickets urgents non détectés

### Comment ça marche ?
L'algorithme isole les points rares dans l'espace
des features — les anomalies sont isolées plus vite.

In [ ]:
# ============================================================
# DÉTECTION ANOMALIES — ISOLATION FOREST
# ============================================================

# Features pour détection anomalies
X_array = X_tfidf.toarray()

# Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,  # 5% anomalies attendues
    random_state=42,
    n_estimators=100
)

df_clean['anomalie'] = iso_forest.fit_predict(X_array)
df_clean['anomalie_score'] = iso_forest.score_samples(X_array)

# -1 = anomalie, 1 = normal
nb_anomalies = (df_clean['anomalie'] == -1).sum()
nb_normaux = (df_clean['anomalie'] == 1).sum()

print(f"=== RÉSULTATS ISOLATION FOREST ===")
print(f"Tickets normaux  : {nb_normaux} ({nb_normaux/len(df_clean)*100:.1f}%)")
print(f"Tickets anormaux : {nb_anomalies} ({nb_anomalies/len(df_clean)*100:.1f}%)")

# Tickets les plus anormaux
df_anomalies = df_clean[
    df_clean['anomalie'] == -1
].sort_values('anomalie_score')

print(f"\n=== TOP 10 TICKETS LES PLUS ANORMAUX ===")
print(df_anomalies[['id', 'objet', 'categorie',
                     'priorite', 'anomalie_score']
].head(10).to_string(index=False))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution scores
axes[0].hist(
    df_clean[df_clean['anomalie'] == 1]['anomalie_score'],
    bins=50, alpha=0.7, color='#1D9E75', label='Normal'
)
axes[0].hist(
    df_clean[df_clean['anomalie'] == -1]['anomalie_score'],
    bins=20, alpha=0.7, color='#E24B4A', label='Anomalie'
)
axes[0].set_title('Distribution scores anomalie')
axes[0].set_xlabel('Score Isolation Forest')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# PCA avec anomalies
axes[1].scatter(
    X_pca[df_clean['anomalie'] == 1, 0],
    X_pca[df_clean['anomalie'] == 1, 1],
    c='#1D9E75', alpha=0.4, s=15, label='Normal'
)
axes[1].scatter(
    X_pca[df_clean['anomalie'] == -1, 0],
    X_pca[df_clean['anomalie'] == -1, 1],
    c='#E24B4A', alpha=0.8, s=40,
    marker='x', label='Anomalie'
)
axes[1].set_title('Anomalies dans l espace PCA')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../diagrams/anomalies_isolation_forest.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SAUVEGARDE MODÈLE ANOMALIES — pour utilisation dans la plateforme
# ============================================================
import pickle
import os

save_dir = '../app/ai_service/models'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/model_anomaly.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
with open(f'{save_dir}/tfidf_anomaly.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("✅ Modèle anomalies sauvegardé : model_anomaly.pkl, tfidf_anomaly.pkl")

---
## 8. Analyse Agents Surchargés

In [ ]:
# ============================================================
# ANALYSE CHARGE AGENTS
# ============================================================
df_raw['agent'] = df_raw['ATTRIBUÉ À']
df_agents = df_raw[df_raw['agent'].notna()].copy()

# Charge par agent par semaine
agent_semaine = df_agents.groupby(
    [df_agents['date_creation'].dt.to_period('W'),
     'agent']
).size().reset_index()
agent_semaine.columns = ['semaine', 'agent', 'nb_tickets']

# Top 5 agents
top_agents = df_agents['agent'].value_counts().head(5).index

# Évolution charge top 5 agents
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Graphique 1 — Évolution par agent
for agent in top_agents:
    data = agent_semaine[
        agent_semaine['agent'] == agent
    ].sort_values('semaine')
    if len(data) > 0:
        axes[0].plot(
            range(len(data)),
            data['nb_tickets'].values,
            marker='o', linewidth=1.5,
            markersize=4, label=agent
        )

axes[0].set_title(
    'Évolution charge des Top 5 agents par semaine',
    fontsize=13, fontweight='bold'
)
axes[0].set_xlabel('Semaine')
axes[0].set_ylabel('Nb tickets')
axes[0].legend(bbox_to_anchor=(1.05, 1),
               loc='upper left', fontsize=9)

# Graphique 2 — Heatmap charge
pivot = agent_semaine[
    agent_semaine['agent'].isin(top_agents)
].pivot_table(
    index='agent',
    columns='semaine',
    values='nb_tickets',
    fill_value=0
)

sns.heatmap(
    pivot,
    cmap='YlOrRd',
    ax=axes[1],
    cbar_kws={'label': 'Nb tickets'},
    linewidths=0.5
)
axes[1].set_title(
    'Heatmap charge agents par semaine',
    fontsize=13, fontweight='bold'
)
axes[1].set_xlabel('Semaine')
axes[1].set_ylabel('Agent')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../../diagrams/charge_agents.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Statistiques
print("=== STATISTIQUES DE CHARGE ===")
stats_df = pd.DataFrame({
    'Agent': df_agents['agent'].value_counts().index,
    'Total tickets': df_agents['agent'].value_counts().values,
    'Moy/semaine': (
        df_agents['agent'].value_counts().values /
        max(len(agent_semaine['semaine'].unique()), 1)
    ).round(1)
})
print(stats_df.head(10).to_string(index=False))

---
## 9. Prédiction SLA Breach (Risque)

Prédire si un ticket risque de dépasser son SLA
basé sur ses caractéristiques.

In [ ]:
# ============================================================
# PRÉDICTION RISQUE SLA BREACH
# ============================================================
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import pickle

# Créer feature de risque SLA
# Un ticket est "à risque" si : P1_URGENT ou P2_HIGH
df_sla = df_clean.copy()
df_sla['sla_risk'] = df_sla['priorite'].apply(
    lambda x: 1 if x in ['P1_URGENT', 'P2_HIGH'] else 0
)

print(f"Tickets à risque SLA : "
      f"{df_sla['sla_risk'].sum()} "
      f"({df_sla['sla_risk'].mean()*100:.1f}%)")

# Features
tfidf_sla = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_sla = tfidf_sla.fit_transform(
    df_sla['text'].fillna('')
)
y_sla = df_sla['sla_risk']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_sla, y_sla,
    test_size=0.2,
    random_state=42,
    stratify=y_sla
)

# Modèle
model_sla = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model_sla.fit(X_tr, y_tr)
y_pred_sla = model_sla.predict(X_te)

from sklearn.metrics import accuracy_score, f1_score
acc = accuracy_score(y_te, y_pred_sla)
f1 = f1_score(y_te, y_pred_sla, average='weighted')

print(f"\n=== MODÈLE RISQUE SLA ===")
print(f"Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"F1-Score : {f1:.4f}")
print(f"\n{classification_report(y_te, y_pred_sla, target_names=['Normal', 'À risque'])}")

# Sauvegarder dans le dossier des modèles IA
save_dir = '../app/ai_service/models'
with open(f'{save_dir}/model_sla_risk.pkl', 'wb') as f:
    pickle.dump(model_sla, f)
with open(f'{save_dir}/tfidf_sla.pkl', 'wb') as f:
    pickle.dump(tfidf_sla, f)

print("✅ Modèle SLA sauvegardé !")

---
## 10. Résumé Final

In [ ]:
# ============================================================
# RÉSUMÉ FINAL
# ============================================================
print("=" * 65)
print("  RÉSUMÉ — ANALYSE PRÉDICTIVE HELPDESK IA GEISER")
print("=" * 65)

print(f"\n📈 PRÉVISION VOLUME (Prophet)")
print(f"   Données        : {len(df_prophet)} jours")
print(f"   Prévision      : 30 jours futurs")
print(f"   Saisonnalités  : hebdomadaire + annuelle")

print(f"\n🔍 CLUSTERING RÉPÉTITIFS (KMeans)")
print(f"   Nb clusters    : {N_CLUSTERS}")
print(f"   Tickets        : {len(df_clean)}")
print(f"   Méthode        : TF-IDF + PCA + KMeans")

print(f"\n⚠️  DÉTECTION ANOMALIES (Isolation Forest)")
print(f"   Tickets normaux  : {nb_normaux}")
print(f"   Tickets anormaux : {nb_anomalies} "
      f"({nb_anomalies/len(df_clean)*100:.1f}%)")

print(f"\n🚨 PRÉDICTION RISQUE SLA")
print(f"   Modèle    : Gradient Boosting")
print(f"   Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
print(f"   F1-Score  : {f1:.4f}")

print(f"\n✅ Graphiques sauvegardés dans diagrams/ :")
print(f"   diagrams/analyse_temporelle.png")
print(f"   diagrams/prevision_prophet.png")
print(f"   diagrams/elbow_kmeans.png")
print(f"   diagrams/clusters_pca.png")
print(f"   diagrams/anomalies_isolation_forest.png")
print(f"   diagrams/charge_agents.png")

print(f"\n✅ Modèles sauvegardés dans app/ai_service/models/ :")
print(f"   models/model_sla_risk.pkl")
print(f"   models/tfidf_sla.pkl")

print(f"\n📋 PHASE 3 — COMPLÈTE !")
print(f"   01 Exploration     ✅  1915 tickets, 10 catégories")
print(f"   02 Classification  ✅  catégorie ~84% / priorité ~91%")
print(f"   03 Routage agent   ✅  ML hybride ~66%")
print(f"   04 RAG ChromaDB    ✅  similarité ≥ 0.60")
print(f"   05 Analyse préd.   ✅  Prophet + KMeans + IsoForest + GBT SLA")

print(f"\n🚀 Service IA intégré dans FastAPI (port 8000)")
print(f"   backend/app/services/ai_service.py")
print(f"   backend/app/routers/ai.py")
print("=" * 65)